# Temas Tratados en el Trabajo Práctico 6

* Modelado de problemas en espacios de estado.

* Algoritmos de planificación hacia adelante y hacia atrás.

* Representación y solución de problemas descritos en lenguaje STRIPS.

* Algoritmo GRAPHPLAN.

* Planificación con restricciones de tiempo y recursos.

* Caminos críticos y tiempos de relajación.

## Ejercicios Teóricos

1. ¿En qué tipo de algoritmos se basa un planificador para encontrar el mejor camino a un estado solución?

Los planificadores se basan en algoritmos de búsqueda en un espacio de estados. Para encontrar el mejor camino, estos planificadores emplean estrategias:

* **Algoritmos informados (como A*):** Utilizan funciones heurísticas, detección de estados repetidos y técnicas de poda para reducir la cantidad de ramas exploradas y encontrar la ruta óptima.


* **Búsqueda direccional:** Pueden operar mediante progresión (buscando hacia adelante, desde el estado inicial aplicando acciones con precondiciones válidas) o mediante regresión (buscando hacia atrás, partiendo del objetivo y regresando a través de las acciones que producen esa meta).


* **Planificación de Orden Parcial (POP):** Algoritmos que manipulan múltiples secuencias y sub-objetivos de manera independiente, estableciendo un orden cronológico entre las acciones solo cuando es estrictamente necesario para evitar conflictos.


2. ¿Qué tres elementos se encuentran dentro de una acción formulada en lenguaje STRIPS? Describa brevemente qué función cumple cada uno.

3. Describa las ventajas y desventajas de desarrollar un algoritmo de planificación hacia adelante y hacia atrás en el espacio de estados.


4. Considere el problema de ponerse uno mismo zapatos y medias. Aplique GRAPHPLAN a este problema y muestre la solución obtenida. Muestre el plan de orden parcial que es solución e indique cuántas linealizaciones diferentes existen para el plan de orden parcial.

In [ ]:
from IPython.display import Image, display

display(Image(filename=r'C:\Users\macie\Pictures\Screenshots\Captura de pantalla 2026-09-24 103421.png'))

In [ ]:
from IPython.display import Image, display

display(Image(filename=r'C:\Users\macie\Pictures\Screenshots\Captura de pantalla 2026-09-24 103433.png'))

5. Se requiere ensamblar una máquina cuyas piezas están identificadas con las letras A, B, C, D y E. El tiempo que se tarda en ensamblar cada pieza es:

* A: 2 semanas

* B: 1 semana

* C: 4 semanas

* D: 3 semanas

* E: 5 semanas

    El orden de ensamblaje de cada pieza requiere que:

* A esté realizado antes que C

* B esté realizado antes que C

* B esté realizado antes que D

* C esté realizado antes que E

* D esté realizado antes que E

    Con esta información:

        5.1 Arme el Plan de Orden Parcial.

        5.2 Encuentre el Camino Crítico.

        5.3 Encuentre los tiempos de relajación.

        5.4 Dibuje un diagrama temporal indicando las tareas y los tiempos de relajación encontrados.

## Ejercicios de Implementación

> Recuerde adjuntar en la presentación el prompt inicial que ha utilizado para cada ejercicio de implementación y si considera que le dio la información completa para resolver el ejercicio o qué cambios adicionales tuvo que pedir de manera iterativa.

6. Suponga que tiene un robot de oficina capaz de moverse y tomar y depositar objetos. El robot solo puede tener un objeto a la vez, pero puede conseguir una *caja* en la que depositar varios objetos. Suponga que programa al robot para *ir a la tienda* a comprarle un *café* y en el camino de vuelta tome una *carta* del *buzón* de la oficina para para que se la traiga junto con el café. Describa en lenguaje STRIPS:

        6.1 El dominio del robot (nombre, predicados y acciones que puede hacer el robot).

        6.2 El problema que se quiere resolver (estado inicial, estado objetivo y objetos del mundo representados).

        6.3 Introduzca el código desarrollado en los puntos anteriores en el [planificador online](http://lcas.lincoln.ac.uk/fast-downward/) y obtenga el plan de acción que tomará el robot para cumplir lo solicitado.

**Ejercicio 6.1**  
*Nombre del dominio:* robot-oficina

*Tipos:* Se definen tres tipos disjuntos. Caja debe ser un tipo distinto de objeto para que las acciones de manipulación de la caja no puedan aplicarse por error sobre café, carta y viceversa.
* localización: lugares (oficina, tienda, buzón)
* objeto: ítems transportables comunes (café, carta)
* caja: contenedor con capacidad múltiple

*Predicados*
|Predicado|Significado|
|:--------|:----------|
|(en-robot ?l - localizacion)|El robot se encuentra en la localización ?l|
|(en-objeto ?o - objeto ?l - localizacion)|El objeto ?o está depositado en el suelo/superficie de ?l|
|(en-venta ?o - objeto ?l - localizacion)|El objeto ?o está disponible para su compra en ?l.|
|(manos-libres)|El efector del robot no sostiene nada (ni objeto ni caja).|
|(sosteniendo ?o - objeto)|El robot sostiene el objeto ?o directamente con su efector.|
|(caja-en ?c - caja ?l - localizacion)|La caja ?c reposa en ?l (no está siendo transportada).|
|(caja-cargada ?c - caja)|El robot transporta consigo la caja ?c.|
|(dentro ?o - objeto ?c - caja)|El objeto ?o está guardado dentro de la caja ?c.|

*Restricción de capacidad*
El predicado (manos-libres) es la clave de la restricción "un objeto a la vez". Nótese que tomar-objeto y tomar-caja requieren y consumen (manos-libres). Por lo tanto, el robot nunca puede estar simultáneamente sosteniendo un objeto y caja-cargada: son estados mutuamente excluyentes. La "trampa" para transportar múltiples objetos es que guardar_en_caja y sacar_de_caja no dependen de manos-libres, sino únicamente de que la caja esté cargada. Así, una vez que el robot carga la caja (ocupando su único "slot" de transporte), puede depositar y extraer objetos de su interior sin volver a liberar las manos.

*Acciones*
|Acciones|Precondiciones|Efectos|
|:-------|:-------------|:------|
|mover(?desde, ?hasta)|(en-robot ?desde)|¬(en-robot ?desde), (en-robot ?hasta)|
|comprar(?o, ?l)|(en-robot ?l), (en-venta ?o ?l)|¬(en-venta ?o ?l), (en-objeto ?o ?l)|
|tomar-objeto(?o, ?l)|(en-robot ?l), (en-objeto ?o ?l), (manos-libres)|¬(en-objeto ?o ?l), ¬(manos-libres), (sosteniendo ?o)|
|dejar-objeto(?o, ?l)|(en-robot ?l), (sosteniendo ?o)|¬(sosteniendo ?o), (manos-libres), (en-objeto ?o ?l)|
|tomar-caja(?c, ?l)|(en-robot ?l), (caja-en ?c ?l), (manos-libres)|¬(caja-en ?c ?l), ¬(manos-libres), (caja-cargada ?c)|
|dejar-caja(?c, ?l)|(en-robot ?l), (caja-cargada ?c)|¬(caja-cargada ?c), (manos-libres), (caja-en ?c ?l)|
|guardar_en_caja(?o, ?c, ?l)|(en-robot ?l), (en-objeto ?o ?l), (caja-cargada ?c)|¬(en-objeto ?o ?l), (dentro ?o ?c)|
|sacar_de_caja(?o, ?c, ?l)|(en-robot ?l), (dentro ?o ?c), (caja-cargada ?c)|¬(dentro ?o ?c), (en-objeto ?o ?l)|

**Ejercicio 6.2**

*Objetos*
* Localizaciones: oficina, tienda, buzón
* Objetos comunes: café, carta
* Contenedor: caja

*Estado inicial*
* El robot está en la oficina, con las manos libres.
* La caja se encuentra disponible en la oficina (aún no ha sido tomada).
* El café está en venta en la tienda (todavía no es un objeto poseído).
* La carta ya se encuentra físicamente en el buzón.

$$I = \{\,en\text{-}robot(oficina),\; manos\text{-}libres,\; caja\text{-}en(caja1,\,oficina),$$
$$en\text{-}venta(cafe,\,tienda),\; en\text{-}objeto(carta,\,buzon),$$
$$conectado(oficina,\,tienda),\; conectado(tienda,\,buzon),\; conectado(buzon,\,oficina)\,\}$$

*Estado objetivo*
* El café debe estar depositado en la oficina.
* La carta debe estar depositada en la oficina.

$$O = \{\,en\text{-}objeto(cafe,\,oficina),\; en\text{-}objeto(carta,\,oficina)\,\}$$

**Ejercicio 6.3**

*Archivo de dominio*

In [ ]:
(define (domain robot-oficina)
  (:requirements :strips :typing)

  (:types
    localizacion
    objeto
    caja
  )

  (:predicates
    (en-robot ?l - localizacion)
    (en-objeto ?o - objeto ?l - localizacion)
    (en-venta ?o - objeto ?l - localizacion)
    (manos-libres)
    (sosteniendo ?o - objeto)
    (caja-en ?c - caja ?l - localizacion)
    (caja-cargada ?c - caja)
    (dentro ?o - objeto ?c - caja)
  )

  (:action mover
    :parameters (?desde - localizacion ?hasta - localizacion)
    :precondition (en-robot ?desde)
    :effect (and (not (en-robot ?desde)) (en-robot ?hasta))
  )

  (:action comprar
    :parameters (?o - objeto ?l - localizacion)
    :precondition (and (en-robot ?l) (en-venta ?o ?l))
    :effect (and (not (en-venta ?o ?l)) (en-objeto ?o ?l))
  )

  (:action tomar-objeto
    :parameters (?o - objeto ?l - localizacion)
    :precondition (and (en-robot ?l) (en-objeto ?o ?l) (manos-libres))
    :effect (and (not (en-objeto ?o ?l)) (not (manos-libres)) (sosteniendo ?o))
  )

  (:action dejar-objeto
    :parameters (?o - objeto ?l - localizacion)
    :precondition (and (en-robot ?l) (sosteniendo ?o))
    :effect (and (not (sosteniendo ?o)) (manos-libres) (en-objeto ?o ?l))
  )

  (:action tomar-caja
    :parameters (?c - caja ?l - localizacion)
    :precondition (and (en-robot ?l) (caja-en ?c ?l) (manos-libres))
    :effect (and (not (caja-en ?c ?l)) (not (manos-libres)) (caja-cargada ?c))
  )

  (:action dejar-caja
    :parameters (?c - caja ?l - localizacion)
    :precondition (and (en-robot ?l) (caja-cargada ?c))
    :effect (and (not (caja-cargada ?c)) (manos-libres) (caja-en ?c ?l))
  )

  (:action guardar_en_caja
    :parameters (?o - objeto ?c - caja ?l - localizacion)
    :precondition (and (en-robot ?l) (en-objeto ?o ?l) (caja-cargada ?c))
    :effect (and (not (en-objeto ?o ?l)) (dentro ?o ?c))
  )

  (:action sacar_de_caja
    :parameters (?o - objeto ?c - caja ?l - localizacion)
    :precondition (and (en-robot ?l) (dentro ?o ?c) (caja-cargada ?c))
    :effect (and (not (dentro ?o ?c)) (en-objeto ?o ?l))
  )
)

*Archivo de problema*

In [ ]:
(define (problem entrega-cafe-carta)
  (:domain robot-oficina)

  (:objects
    oficina tienda buzon - localizacion
    cafe carta - objeto
    caja1 - caja
  )

  (:init
    (en-robot oficina)
    (manos-libres)
    (caja-en caja1 oficina)
    (en-venta cafe tienda)
    (en-objeto carta buzon)
  )

  (:goal
    (and
      (en-objeto cafe oficina)
      (en-objeto carta oficina)
    )
  )
)

*Plan*

In [2]:
display(Image(filename=r'C:\Users\valel\Downloads\plan_tp6_ia1.png'))

NameError: name 'Image' is not defined


# Bibliografía

[Russell, S. & Norvig, P. (2004) _Inteligencia Artificial: Un Enfoque Moderno_. Pearson Educación S.A. (2a Ed.) Madrid, España](https://www.academia.edu/8241613/Inteligencia_Aritificial_Un_Enfoque_Moderno_2da_Edici%C3%B3n_Stuart_J_Russell_y_Peter_Norvig)

[Poole, D. & Mackworth, A. (2023) _Artificial Intelligence: Foundations of Computational Agents_. Cambridge University Press (3a Ed.) Vancouver, Canada](https://artint.info/3e/html/ArtInt3e.html)